In [ ]:
from IROS._pipeline_support import _handle_dirpaths

import numpy as np
import pandas as pd

from bloodmoon.io import simulation_files
from bloodmoon.mask import codedmask

import darksun as ds

In [ ]:
skyfield = "GalacticCenter"

#mask_FITS = "wfm_mask_summer2021.fits"
#data_FITS = "galctr_rxte-sax_mask_summer2021_infdet_2-50keV_1ks"

mask_FITS = "wfm_mask_NTHT_20250725.fits"
data_FITS = "galctr_rxte-sax_mask_050_1040x17_infdet_2-50keV_1ks"

#skyfield = "IROSDummy"
#data_FITS = "catalog_withCXB_1Crab_infdet_2-50keV_1ks"

N_TEST = "testing"

mask_path, simul_data, save_path = _handle_dirpaths(
    mask=mask_FITS,
    skyfield=skyfield,
    simul=data_FITS,
    run_name=N_TEST,
)

VIGNETTING = True
PSFY = False
UPX, UPY = 5, 1
wfm = codedmask(mask_path, UPX, UPY)

cam_a = "cam1a"
cam_b = "cam1b"
dataset = 'detected'

filepaths = simulation_files(simul_data)
sdlA = ds.get_data(filepaths[cam_a][dataset])
catalogueA = ds.get_catalogue(filepaths[cam_a]['sources'])

simul_sky_camA, _ = ds.load_sky(save_path + f"sky_SIMUL_CAM1A_TEST_{N_TEST}.fits")
simul_sky_camB, _ = ds.load_sky(save_path + f"sky_SIMUL_CAM1B_TEST_{N_TEST}.fits")
log_camA, log_camB = ds.load_database(save_path + f"IROS_sources_database_TEST_{N_TEST}.fits")

In [ ]:
pd.DataFrame(
    {
        log_camA.name: log_camA.log['ID'],
        f"{log_camA.name} SNR": log_camA.log['snr'],
        log_camB.name: log_camB.log['ID'],
        f"{log_camB.name} SNR": log_camB.log['snr'],
        "Same Source": np.array(log_camA.log['ID']) == np.array(log_camB.log['ID']),
    }
)

In [ ]:
from typing import NamedTuple

from bloodmoon.mask import CodedMaskCamera
from bloodmoon.types import CoordEquatorial
from bloodmoon.coords import equatorial2shift, shift2angle

from darksun.data import Log, DataLoader, CatalogueLoader

class CoordAngular(NamedTuple):
    """
    Local frame coded-mask camera angular coordinates.

    Args:
        theta_x (float): Angular coord along x-axis [deg]
        theta_y (float): Angular coord along y-axis [deg]
    """
    theta_x: float
    theta_y: float

def extract_catalogue_equatorial_coords(
    log: Log,
    catalogue: CatalogueLoader,
) -> tuple[tuple[str, CoordEquatorial]]:
    """
    """
    sources, ras, decs = [], [], []
    for s_id in log.log['ID']:
        if s_id in catalogue.DLdata['ID']:
            source = catalogue.DLdata[(catalogue.DLdata['ID'] == s_id)]
            sources.append(s_id)
            ras.append(source['RA'])
            decs.append(source['DEC'])
        else:
            print(f'Source {s_id.upper()} not in catalogue...')
            sources.append(s_id)
            ras.append(-999)
            decs.append(-999)
    return tuple(
        (name, CoordEquatorial(ra, dec)) for name, ra, dec in zip(sources, ras, decs)
    )

def extract_catalogue_angular_coords(
    log: Log,
    catalogue: CatalogueLoader,
    sdl: DataLoader,
    camera: CodedMaskCamera,
) -> tuple[tuple[str, CoordAngular]]:
    """
    """
    sources, thetas_x, thetas_y = [], [], []
    for s_id in log.log['ID']:
        if s_id in catalogue.DLdata['ID']:
            source = catalogue.DLdata[(catalogue.DLdata['ID'] == s_id)]
            sources.append(s_id)
            sx, sy = equatorial2shift(sdl, camera, source['RA'], source['DEC'])
            thetas_x.append(shift2angle(camera, sx))
            thetas_y.append(shift2angle(camera, sy))
        else:
            print(f'Source {s_id.upper()} not in catalogue...')
            sources.append(s_id)
            thetas_x.append(-999)
            thetas_y.append(-999)
    return tuple(
        (name, CoordAngular(tx, ty)) for name, tx, ty in zip(sources, thetas_x, thetas_y)
    )


_, cat_theta = zip(
    *extract_catalogue_angular_coords(log=log_camA, catalogue=catalogueA, sdl=sdlA, camera=wfm)
)
cat_thetax, cat_thetay = zip(*cat_theta)

dmapx = ds.map4biplot(
    arrs=60 * np.abs(np.array(log_camA.log['angle_x']) - np.array(cat_thetax)),
    title=f'{log_camA.name} X-Axis Angular Residues (abs values)',
    xlabel='$\\theta_{{x}}$ off-axis angle [deg]',
    ylabel='$\\Delta\\theta_{{x}}$ residues [arcmin]',
    x=cat_thetax,
    style='scatter',
    xlim=(-45, 45),
    yscale='log',
    tags=tuple(
        (n, r, t) for n, (r, t) in enumerate(tuple(zip(
            60 * np.abs(np.array(log_camA.log['angle_x']) - np.array(cat_thetax)), cat_thetax,
        )))
    ),
)
dmapy = ds.map4biplot(
    arrs=60 * np.abs(np.array(log_camA.log['angle_y']) - np.array(cat_thetay)),
    title=f'{log_camA.name} Y-Axis Angular Residues (abs values)',
    xlabel='$\\theta_{{y}}$ off-axis angle [deg]',
    ylabel='$\\Delta\\theta_{{y}}$ residues [arcmin]',
    x=cat_thetay,
    style='scatter',
    xlim=(-45, 45),
    yscale='log',
    tags=tuple(
        (n, r, t) for n, (r, t) in enumerate(tuple(zip(
            60 * np.abs(np.array(log_camA.log['angle_y']) - np.array(cat_thetay)), cat_thetay,
        )))
    ),
)
ds.biplot(dmapx, dmapy)

In [ ]:
ds.skyfield_map(log_camA, wfm)
ds.skyfield_map(log_camB, wfm)

In [ ]:
cropx, cropy = (
    int(wfm.specs["slit_deltax"] * UPX / wfm.specs["mask_deltax"] + 5),
    int(wfm.specs["slit_deltay"] * UPY / wfm.specs["mask_deltay"] + 5),
)
ds.reconstruction_plot(
    true_sky=simul_sky_camA,
    log=log_camA,
    crp=(cropy, cropx),
    camera=wfm,
    vignetting=VIGNETTING,
    psfy=PSFY,
)